# Opti-GEMM — CUDA GEMM on Colab

This notebook is a compact GPU lab for the [Opti-GEMM](https://github.com/Danialjfz/Opti-GEMM) project: four handwritten CUDA GEMM kernels measured against cuBLAS on a free GPU.

It walks through the full workflow:
1. Build the CUDA project on a free Colab GPU (Tesla T4)
2. Validate every kernel against a CPU reference
3. Benchmark the Naive, Tiled-SMEM, Reg-Block, and Warp-Tile kernels
4. Compare all four against cuBLAS (full FP32, pedantic math)
5. Profile the kernels with Nsight Compute and export a `.ncu-rep`

Typical Colab runtime: `Runtime → Change runtime type → GPU (T4)`.

Reference T4 numbers for every step live in the [README](https://github.com/Danialjfz/Opti-GEMM/blob/develop/README.md): on large shapes Warp-Tile sustains ~2.4–2.6 TFLOP/s, reaching 0.70–0.79× cuBLAS.

> The notebook clones the `develop` branch, where kernel work lands first (`main` is updated from `develop` at milestones).

## 0. Check the GPU and CUDA toolchain

This confirms the environment is ready before building and profiling.


In [ ]:
!nvidia-smi
!nvcc --version | tail -2
# ncu ships with the CUDA toolkit preinstalled on Colab:
!which ncu && ncu --version | head -3 || echo "ncu NOT on PATH — see troubleshooting at the bottom"

## 1. Clone and build

Clones the `develop` branch (where the current kernel code lands first) and builds with `CMAKE_CUDA_ARCHITECTURES=native`, which automatically targets the detected Colab GPU (for example T4 → `75`, A100 → `80`, L4 → `89`).

In [ ]:
!git clone --depth 1 --branch develop https://github.com/Danialjfz/Opti-GEMM.git
%cd Opti-GEMM
!cmake -S . -B build -DOPTI-GEMM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

### 1b. Register and shared-memory footprint (`ptxas`)

The README's occupancy analysis cites per-thread register counts and per-block shared-memory usage for the Reg-Block and Warp-Tile kernels. Compiling both files with `--ptxas-options=-v` prints exactly those numbers — the two values that set each kernel's occupancy ceiling (Reg-Block: ~72 registers and 33,280 B of shared memory per block; Warp-Tile's 64 accumulators per thread push register pressure higher by design).

In [ ]:
# ptxas verbose output goes to stderr; Colab shows it inline.
# (-arch=native needs CUDA >= 11.6; on an older toolkit use -arch=sm_75)
!nvcc -c src/cuda/gemm_regblock.cu -o /tmp/regblock.o -Iinclude -std=c++17 -arch=native --ptxas-options=-v
!nvcc -c src/cuda/gemm_warptile.cu -o /tmp/warptile.o -Iinclude -std=c++17 -arch=native --ptxas-options=-v

## 2. Correctness check

Runs every kernel against the CPU reference across representative problem sizes, including edge cases and non-square shapes. A kernel that fails here must not be trusted in the benchmark.


In [ ]:
!./build/correctness

## 3. Kernel benchmark: Naive → Tiled-SMEM → Reg-Block → Warp-Tile

`bench_kernels` validates each kernel against the CPU reference before timing it (a kernel that fails is skipped), then reports median/mean/stddev and GFLOP/s across sizes 128³–4096³.

> On a T4, the Naive kernel is often surprisingly competitive because Turing's unified L1 already caches global loads. Watch the arithmetic-intensity ladder pay off as the problem grows: Reg-Block reuses each loaded value across a 4×4 micro-tile (16 FLOP/byte per block tile), and Warp-Tile doubles the block tile to 128×128 with an 8×8 micro-tile (32 FLOP/byte, 64 accumulators per thread), overtaking Reg-Block from 512³ onward.

In [ ]:
!./build/bench_kernels

## 4. cuBLAS comparison

`bench_vs_cublas` validates the Naive, Tiled-SMEM, Reg-Block, Warp-Tile, and cuBLAS results against the CPU reference before timing them with CUDA events. The output reports median/mean/stddev latency, GFLOP/s, and each implementation's speed relative to cuBLAS.

The handwritten kernels and cuBLAS use the same row-major problem layout. Internally, the cuBLAS call computes the equivalent column-major `Cᵀ = Bᵀ × Aᵀ` operation. The benchmark selects `CUBLAS_PEDANTIC_MATH` so the comparison uses full FP32 arithmetic rather than reduced-precision tensor-core math.

This step is intentionally separate from Nsight profiling: use its unprofiled output for performance numbers, because Nsight Compute replay changes timing.


In [ ]:
!./build/bench_vs_cublas

### Expected results on a Tesla T4

Reference numbers from the README's T4 run of this exact binary (same warmup/sample structure). Small run-to-run variation is normal; the ranking should be stable — Warp-Tile should land at 0.70–0.79× cuBLAS on every sufficiently large shape:

| Shape | Naive | Tiled-SMEM | Reg-Block | Warp-Tile | cuBLAS |
|---|---|---|---|---|---|
| 512³ | 668 | 675 | 1,477 | 1,598 | 5,382 |
| 1024³ | 584 | 626 | 1,598 | 2,516 | 3,612 |
| 2048³ | 463 | 503 | 1,556 | 2,607 | 3,424 |
| 4096³ | 371 | 456 | 1,499 | 2,443 | 3,415 |

(GFLOP/s, median of 7 samples × 100 repeats after 10 warmup iterations; cuBLAS in `CUBLAS_PEDANTIC_MATH` FP32 mode.)

The suite also covers odd and non-square shapes (1000³, 1023 × 1025 × 1027, 4096 × 128 × 1024, 128 × 4096 × 1024, 1024 × 1024 × 32) to exercise every kernel's bounds-checked edge paths. Warp-Tile's best relative result — 0.79× cuBLAS — comes from the fully irregular 1023 × 1025 × 1027 case.

## 5. Profile with Nsight Compute (`ncu`)

`ncu` launches the target binary and collects kernel-level metrics while the program runs. The examples below focus on the most useful profiling views for a GEMM kernel.

Kernel symbol names used in the filters below: `naive_gemm`, `tiled_gemm`, `regblock_gemm_kernel`, `warptile_gemm_kernel`.

Key flags:

| Flag | Meaning |
|---|---|
| `--kernel-name regex:"..."` | profile only matching kernel names |
| `--launch-skip N` | skip warmup launches |
| `--launch-count N` | collect metrics for the next N launches |
| `--section SpeedOfLight` | summarize compute vs. memory utilization |
| `--set detailed / full` | use curated metric bundles |
| `--export file` | write a `.ncu-rep` file for the desktop GUI |

**Launch arithmetic used below:** `bench_kernels` launches each kernel **711 times per size** — 1 correctness validation + 10 warmup + 700 timed (7 samples × 100 repeats) — across 6 sizes (128³ … 4096³). With `--kernel-name` filtering, `--launch-skip` counts only matching launches, so the first steady-state timed launch of a kernel at size index `i` (0-based) is `i × 711 + 11`: **1433** for 512³, **2144** for 1024³, **3566** for 4096³.

> Important caveats:
> 1. Do not trust the timed rows from a profiled run. `ncu` replays the profiled launch multiple times, which inflates the reported timing. Use the unprofiled runs from section 3 (and the cuBLAS comparison in section 4) for the trustworthy numbers.
> 2. `ncu` often locks clocks to the base frequency (`--clock-control base`), so profiled times can look slower than real wall-clock measurements. Use `--clock-control none` if you want more realistic timing.
> 3. Every `ncu` invocation re-runs the whole benchmark from the start — including the single-threaded CPU reference — so the large-size cells below can take several minutes each.

### 5a. Speed-of-Light at 512³: compute-bound or memory-bound?

A quick top-level read on whether each kernel is limited by arithmetic throughput or memory traffic. 512³ is the size the README's SOL table was captured at, so these numbers are directly comparable to it.

In [ ]:
# First steady-state timed launches at 512³: skip = 2 × 711 + 11 = 1433
!ncu --section SpeedOfLight --kernel-name naive_gemm \
     --launch-skip 1433 --launch-count 2 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --kernel-name tiled_gemm \
     --launch-skip 1433 --launch-count 2 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --kernel-name regblock_gemm_kernel \
     --launch-skip 1433 --launch-count 2 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --kernel-name warptile_gemm_kernel \
     --launch-skip 1433 --launch-count 2 \
     ./build/bench_kernels

### 5b. The 4096³ case: when DRAM finally matters

At smaller sizes the working set stays L1/L2-resident and even the Naive kernel looks fine. At 4096³ the matrices (~192 MB total) blow past the 4 MB L2, and the cost shifts to DRAM traffic.

4096³ is the 6th size (index 5), so steady-state timed launches start at `skip = 5 × 711 + 11 = 3566`.

What to look for:
- DRAM throughput rising sharply compared to the 512³ profiles
- L2 hit rate collapsing as data no longer fits in cache
- `Executed IPC` and `Compute Workload Analysis` showing whether the FP32 FMA pipes are actually busy
- Warp-Tile should show the highest compute (SM) throughput of the handwritten kernels while keeping DRAM traffic in check — that is the 32 FLOP/byte arithmetic intensity paying off

> This cell runs the benchmark four times under `ncu` (once per kernel), each time up to the 4096³ launches — expect it to take a while. Comment out lines you don't need.

In [ ]:
!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name naive_gemm --launch-skip 3566 --launch-count 1 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name tiled_gemm --launch-skip 3566 --launch-count 1 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name regblock_gemm_kernel --launch-skip 3566 --launch-count 1 \
     ./build/bench_kernels

!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section ComputeWorkloadAnalysis \
     --kernel-name warptile_gemm_kernel --launch-skip 3566 --launch-count 1 \
     ./build/bench_kernels

### 5c. Why is it slow? Memory traffic and warp stalls

These sections explain the bottleneck rather than just reporting the symptom:
- `MemoryWorkloadAnalysis`: L1/L2/DRAM traffic and redundant global reads
- `WarpStateStats`: stall reasons such as `stall_long_scoreboard` (waiting on memory)

This is where the Naive kernel's 2·K global loads per output element become obvious — and where Warp-Tile's deliberate low-occupancy, high-ILP design (64 accumulators per thread) can be checked against Reg-Block and Tiled-SMEM.

In [ ]:
!ncu --section MemoryWorkloadAnalysis --section WarpStateStats \
     --kernel-name naive_gemm --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

!ncu --section MemoryWorkloadAnalysis --section WarpStateStats \
     --kernel-name tiled_gemm --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

!ncu --section MemoryWorkloadAnalysis --section WarpStateStats --section Occupancy \
     --kernel-name regblock_gemm_kernel --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

!ncu --section MemoryWorkloadAnalysis --section WarpStateStats --section Occupancy \
     --kernel-name warptile_gemm_kernel --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

### 5d. Shared-memory bank conflicts

All three shared-memory kernels pad against bank conflicts: Tiled-SMEM and Reg-Block pad both tiles to `[TILE][TILE+1]`, while Warp-Tile pads only `As` to 128×9 — its `Bs` is 128 columns wide, so column reads already spread across all 32 banks. This check should report ~0 conflicts for all of them — a good sanity check on the optimized paths.

In [ ]:
!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,\
l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
sm__warps_active.avg.pct_of_peak_sustained_active \
     --kernel-name tiled_gemm --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,\
l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
sm__warps_active.avg.pct_of_peak_sustained_active \
     --kernel-name regblock_gemm_kernel --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,\
l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
sm__warps_active.avg.pct_of_peak_sustained_active \
     --kernel-name warptile_gemm_kernel --launch-skip 1433 --launch-count 1 \
     ./build/bench_kernels

### 5e. Export a full profile for desktop analysis

Use `--set full` when you want the complete report. Keep the launch count small because `ncu` replays each profiled launch and the full set is intentionally expensive. The example below captures one fully-profiled launch of `warptile_gemm_kernel` at 1024³ — its first size above 2.5 TFLOP/s, at `skip = 3 × 711 + 11 = 2144`. Swap the kernel name (and skip count, per the arithmetic in section 5) for another kernel or size.

Download the `.ncu-rep` and open it in Nsight Compute on a local machine to inspect the roofline chart, instruction-level activity, and source/SASS correlation. Record what you find in `docs/kernel_analysis.md` (see `profiling/nsight_notes.md` §7 for what to capture).

In [ ]:
!ncu --set full --force-overwrite \
     --export /content/opti_gemm_warptile_full \
     --kernel-name warptile_gemm_kernel \
     --launch-skip 2144 --launch-count 1 \
     ./build/bench_kernels

from google.colab import files
files.download('/content/opti_gemm_warptile_full.ncu-rep')

## Troubleshooting

- `ncu` not found: it is commonly installed under `/usr/local/cuda/bin`. Try `!ls /usr/local/cuda*/bin/ncu` and use the full path if needed, or install the CUDA toolkit package for your Ubuntu release.
- `ERR_NVGPUCTRPERM`: the VM blocks performance counters. On Colab, restarting the runtime usually fixes it; on Kaggle (T4) counters are blocked entirely — plain benchmarks still run, but the section-5 `ncu` cells will fail there. Use Colab for profiling.
- Old code after cloning? The notebook pins `--branch develop`; check `!git log --oneline -1` inside the cloned folder.
- Skip counts off? They assume every kernel passes correctness in `bench_kernels`. A kernel that fails validation is skipped, removing its 711 launches per size for every later kernel in the table — re-check the section 3 output.
- Want a P100 to reproduce the README's Pascal numbers? Kaggle notebooks are a good alternative because they often provide a free P100 with the CUDA tools already available.
- Need timeline profiling? `nsys` is often installed as well: `!nsys profile -o timeline ./build/bench_kernels`.